# Comparing Ansible execution modes: ansible-playbook, ansible-navigator, AAP job templates, ansible-rulebook

Four ways to run the same automation, and when each one earns its place: direct CLI runs, containerized navigator runs, control-plane job templates, and event-driven rulebooks.

> **Scope** — written against ansible-core 2.21.5 (published 2026-10-05, current GA line), with the 2.22 release timeline as the upgrade-planning reference.

---

**last_verified:** 2026-10-05  
**tool_version:** ansible-core 2.21.5  
**sources:** https://api.github.com/repos/ansible/ansible/releases?per_page=3 https://docs.ansible.com/projects/ansible-core/devel/roadmap/ROADMAP_2_22.html https://raw.githubusercontent.com/ansible/ansible/v2.22.0b2/changelogs/CHANGELOG-v2.22.rst https://api.github.com/repos/ansible/awx-operator/releases/latest 

## Purpose

A playbook describes desired state; an execution mode decides where, how, and on whose authority it runs. The four modes differ along four axes: trigger (human, schedule, or event), runtime (local Python environment vs containerized execution environment), access control (shell access vs role-based launch permissions), and auditability (terminal scrollback vs a stored job record). This notebook maps each mode to the axis values it provides so the choice becomes a lookup, not a debate.

## When to use

- **ansible-playbook** — developing a playbook, debugging a failure, or running a one-off change from a workstation or a CI step where the caller already holds credentials.
- **ansible-navigator** — the run must be reproducible across machines: same execution environment, same collection set, same behavior on a laptop and in CI.
- **AAP / AWX job template** — the run needs a schedule, delegated launch rights (someone triggers without shell access), prompted inputs, or a retained audit record. The self-managed path is AWX alongside operator 2.19.1 / AWX v24.6.1.
- **ansible-rulebook** — no human starts the run. An event source (webhook, alert, message) satisfies a condition, and the matching action launches automation in response.

## Prerequisites

- ansible-core installed. The current GA pin is 2.21.5 (published 2026-10-05, alongside 2.20.10 and 2.19.14 on the older stable lines); pin the 2.21.x line for production and keep 2.22 pre-releases in staging until the 2026-11-02 release lands.
- An inventory and at least one playbook under version control.
- For navigator: a container runtime and an execution-environment image. For job templates: access to an AAP instance or an AWX deployment. For rulebooks: a reachable event source.

## 1. ansible-playbook — the direct run

The CLI executes a playbook file against an inventory in the current shell, using the local Python environment and whatever credentials the caller holds. It is the fastest feedback loop and therefore the development default. `--check` previews changes without applying them; `--limit` scopes the run to a subset of hosts.

In [ ]:
ansible-playbook --version 2>/dev/null | head -1 || echo "ansible-playbook not installed in this kernel - shapes below are illustrative."
echo '---'
echo 'ansible-playbook site.yml -i inventory.ini --limit web --check'

## 2. ansible-navigator — the reproducible run

Navigator wraps the same playbook execution in a containerized runtime. The playbook, collections, and Python dependencies travel inside an execution environment image, so a run on a laptop behaves like the run in CI. Use it when "works on my machine" disputes appear, or when the control node cannot (or should not) carry every collection natively. The trade-off is image build and distribution overhead in exchange for parity.

In [ ]:
ansible-navigator --version 2>/dev/null | head -1 || echo "ansible-navigator not installed in this kernel - shape below is illustrative."
echo '---'
echo 'ansible-navigator run site.yml -i inventory.ini'

## 3. AAP / AWX job templates — the governed run

A job template binds together inventory, project (the version-controlled playbook source), credentials, and launch options into a named, launchable object on the control plane. Templates add what the CLI cannot: schedules, role-based launch permissions for users without shell access, prompted fields for per-run inputs, and a retained record of every launch with its output. The self-managed equivalent is AWX (operator 2.19.1, released with AWX v24.6.1). Choose templates when the question "who ran what, when, against which hosts" must have an answer after the fact.

## 4. ansible-rulebook — the event-driven run

A rulebook inverts control: instead of a human or a clock starting the run, a source emits events, rules match conditions against those events, and actions (typically launching a playbook) fire when a condition holds. The cell below sketches the shape — a webhook source, one condition, one playbook action — without touching any live system.

In [ ]:
cat > /tmp/rulebook-sketch.yml <<'EOF'
---
- name: Remediate on alert webhook
  hosts: all
  sources:
    - name: alert webhook
      # event source plugin goes here (e.g. a webhook listener)
  rules:
    - name: disk-pressure alert fires remediation
      condition: event.payload.alert == "disk_pressure"
      action:
        run_playbook:
          name: remediate-disk.yml
EOF
echo "wrote /tmp/rulebook-sketch.yml:"
cat /tmp/rulebook-sketch.yml

## Comparison

| Dimension | ansible-playbook | ansible-navigator | Job template (AAP/AWX) | ansible-rulebook |
|---|---|---|---|
| Trigger | Human at a shell / CI step | Human, container-wrapped | Human, schedule, or API launch | Event matching a condition |
| Runtime | Local Python environment | Execution-environment container | Execution nodes managed by the platform | Rulebook process plus launched playbooks |
| Credentials held by | The caller | The caller | The platform (RBAC-scoped) | The rulebook service account |
| Audit record | Terminal output only | Terminal output only | Stored per-launch job record | Event log plus launched-job records |
| Reproducibility | Depends on the caller's machine | High (image-pinned) | High (platform-pinned) | Depends on pinned content |
| Best for | Development, debugging, one-offs | Parity across laptop and CI | Delegation, schedules, compliance | Hands-off remediation and webhooks |

In [ ]:
pick_mode() {
    case "$1" in
        cli|adhoc|debug)        echo "ansible-playbook: direct CLI run against an inventory" ;;
        reproduce|ee|parity)    echo "ansible-navigator: containerized run for cross-machine parity" ;;
        schedule|rbac|survey|audit) echo "job template: scheduled, permission-scoped, audited run" ;;
        event|webhook|alert)     echo "ansible-rulebook: event condition launches the playbook" ;;
        *)                      echo "unknown scenario: default to ansible-playbook, then re-evaluate" ;;
    esac
}
for s in debug parity audit alert typo; do
    printf '%-8s -> %s\n' "$s" "$(pick_mode "$s")"
done

## Verify

1. **Helper:** run the `pick_mode` cell — `debug`, `parity`, `audit`, `alert` each resolve to a distinct mode and anything else falls through to the playbook default.
2. **Playbook parity:** run the same playbook with `--check` first, then for real; the check run must report the same change set the real run applies.
3. **Version pin:** `ansible-playbook --version` reports a 2.21.x core on production paths; 2.22 material (Feature Freeze 2026-09-14, Beta 1 2026-09-21, RC 1 2026-10-12, Release 2026-11-02) stays in staging until GA.
4. **Config behavior:** after any 2.22 upgrade, confirm `ansible-config` scoping — the 2.22 line defaults to listing all actions (`-t all`; `-t base` restores the old view) — before trusting a config diff across modes.

## Common errors

- **Developing against one mode, deploying on another:** a playbook that passes locally under ansible-playbook fails on the platform because a collection present on the laptop is missing from the execution environment. Promote through navigator before templating.
- **Leaking secrets into job output:** the 2.22 line changed redaction behavior (`no_log` values are kept and masked rather than replaced with a placeholder token, and `run_command` no longer rewrites password-like args). Re-check what old audit filters assumed about placeholder strings after upgrading.
- **Rulebook fires twice:** overlapping conditions match the same event. Keep conditions mutually exclusive or make the remediation playbook idempotent so a double fire is harmless.
- **Schedule sprawl:** one template per host group per hour becomes unmaintainable. Prefer fewer templates with prompted limits over a combinatorial explosion of near-identical templates.

## References

- Current GA and stable lines (2.21.5, 2.20.10, 2.19.14, all 2026-10-05): https://api.github.com/repos/ansible/ansible/releases?per_page=3 
)- 2.22 release-phase schedule and planned/delayed work: https://docs.ansible.com/projects/ansible-core/devel/roadmap/ROADMAP_2_22.html 
)- 2.22 breaking changes and deprecation watch-list (incl. CVE-2026-11332 identifier; the changelog publishes no CVSS score or version range for it): https://raw.githubusercontent.com/ansible/ansible/v2.22.0b2/changelogs/CHANGELOG-v2.22.rst 
)- AWX operator release state (2.19.1 with AWX v24.6.1): https://api.github.com/repos/ansible/awx-operator/releases/latest